# Lab 1 — Limit order book data with LOBSTER

**COMP0049 Market Microstructure · MSc Computational Finance, UCL · 2026–27**

**Learning outcomes.** By the end of the session you should be able to

1. set up Python and use the main `pandas` functions needed for high-frequency data;
2. load and analyse limit order book (LOB) data in the LOBSTER format;
3. build indicators of **liquidity** (e.g. depth) and of **risk / cost of trading** (e.g. spread) from LOB data;
4. compare the empirical properties of different classes of stocks (**small-tick** vs **large-tick**).

**Theory.** Lecture notes Ch. 1–3 (LOB definitions, the tick size, empirical properties) and §8.1 (the LOBSTER dataset and the Lab 1 exercises).

| Part | What we do | Lecture notes | 
|---|---|---|
| 0 | Setting up Python, finding the data| — |
| 1 | The **message file**: events, event types, halts, executions vs trades, hidden liquidity | §2.4 |
| 2 | The **order book file**: best quotes, spread, mid-price, gaps, a snapshot of the book | §2.1, §3.1.2 | 
| 3 | Event time vs clock time | - | 5 |
| 4 | Liquidity and risk indicators for a **small-tick** and a **large-tick** stock | §2.5, §3.1 | 
| 5 | Different classes of stocks on 6 October 2025 | Fig. 3.10 | 
| 6 | Selective liquidity taking: the consumption ratio | §3.1.5 |

## Data access and rules

**1. LOBSTER (NASDAQ academic data).** The course has a shared student account:

- https://php.lobsterdata.com/index.php
- Account: `studentlobsterucl@gmail.com`
- Password: `Ucl_lobster`

Please use it responsibly, as the account is shared with the whole class and can be suspended for all of us:

- **No bulk downloads.** Request only the stock days you need for the labs or for your project, one request at a time.
- **Read the terms of use and the data request rules** on the LOBSTER site before submitting a request.
- **No redistribution.** The data are for academic use within this module only: do not share, publish or resell them.
- **If the site stops responding** or a download hangs, clear your browser cache and cookies (or use a private window) and log in again.

**2. Other data for summer projects (FX, trades & quotes).** Dukascopy offers free historical data (tick-level FX quotes, among others):

- https://www.dukascopy.com/swiss/english/marketwatch/historical/

Download the files **manually** from the website: do not scrape the site or automate the downloads. You can build the dataset for your project by hand, one instrument and period at a time.

> **Be compliant.** Every data provider has terms of use. Breaking them (bulk or automated downloads, redistribution) can get the course account suspended and is a disciplinary matter. If in doubt, ask before downloading.

LOBSTER (https://lobsterdata.com) reconstructs the NASDAQ limit order book from the exchange's official TotalView-ITCH feed. For each stock and day we get **two files with the same number of rows**:

- `TICKER_DATE_34200000_57600000_message_10.csv` — one row per **event** (a submission, cancellation, deletion or execution);
- `TICKER_DATE_34200000_57600000_orderbook_10.csv` — the **state of the book** (10 best price levels on each side) immediately *after* that event.

`34200000` and `57600000` are the start and end of the sample in milliseconds after midnight, i.e. 09:30 and 16:00 New York time; `10` is the number of price levels.

| Stocks | Trading day | Used in |
|---|---|---|
| AMZN, MSFT | 21 June 2012 (LOBSTER's free sample) | Parts 1–4|
| TSLA, GOOG, BAC, BKNG | 6 October 2025 | Parts 5–6 |

**Where to put the files.** Download the 10-level files from the module SharePoint (link on Moodle) and save them in a folder called `data` next to this notebook. The AMZN and MSFT samples can also be downloaded for free from https://lobsterdata.com/info/DataSamples.php.

**Reference.** Bouchaud, Bonart, Donier & Gould, *Trades, Quotes and Prices*, CUP 2018 (free through the UCL library on Cambridge Core).

---
# Part 0 — Setting up Python

## 0.1 Libraries

We only need the standard scientific stack: `numpy` (arrays), `pandas` (tables and time series) and `matplotlib` (plots). If an import fails, install the package from a terminal with `pip install numpy pandas matplotlib` and restart the kernel.

In [1]:
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, MaxNLocator, NullFormatter

print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {mpl.__version__}")

pandas 2.3.3 | numpy 2.3.5 | matplotlib 3.10.6


In [2]:
# Constants used throughout the lab 
START_TRAD  = 9.5 * 60 * 60     # 09:30:00 in seconds after midnight (market open)
END_TRAD    = 16.0 * 60 * 60    # 16:00:00 in seconds after midnight (market close)
PRICE_SCALE = 10_000            # LOBSTER stores prices as dollars x 10,000  ($51.40 -> 514000)
TICK        = 100               # tick size tau = $0.01 = 100 LOBSTER price units (US stocks above $1)
EMPTY_LEVEL = 9_999_999_999     # LOBSTER's filler price for an unoccupied level (negative on the bid side)

# Plot style: one colour per meaning, used consistently 
BID_COLOR, ASK_COLOR = "#2a78d6", "#eb6834"            # bid side = blue, ask side = orange
VISIBLE_COLOR, HIDDEN_COLOR = "#52514e", "#eda100"     # visible executions = grey, hidden = amber
STOCK_PALETTE = ["#6250d6", "#1baf7a"]                  
ACCENT, MUTED, LIGHT = "#2a78d6", "#898781", "#c3c2b7"

plt.rcParams.update({
    "figure.figsize": (10, 4), "figure.dpi": 100, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": LIGHT, "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "legend.frameon": False,
})


def fmt(table, decimals=2):
    """Format the numbers of a summary table for display: thousands separators and a fixed number of decimals."""
    def one(v):
        if isinstance(v, (bool, np.bool_, str)):
            return v
        if isinstance(v, (int, np.integer)):
            return f"{v:,}"
        if isinstance(v, (float, np.floating)):
            return f"{v:,.{decimals}f}"
        return v
    return table.apply(lambda column: column.map(one))


def hours(index):
    """Time of day in hours (9.5 = 09:30) for a DatetimeIndex, so different trading days share one axis."""
    return (index - index.normalize()) / pd.Timedelta(hours=1)


def time_axis(ax):
    """Format an x-axis expressed in hours-of-day as 09:30 ... 16:00."""
    ax.set_xlim(9.5, 16)
    ax.set_xticks(range(10, 17))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda h, _: f"{int(h):02d}:{int(round((h % 1) * 60)):02d}"))
    ax.set_xlabel("Time of day (New York)")

## 0.2 Where is the data?

The next cell looks for the LOBSTER files in a few standard places and stops at the first folder that contains them:

1. a folder you set by hand in `MY_DATA_DIR`;
2. `data/` next to this notebook;
3. `Microstructure demos/LOBdemo25/data/` next to this notebook.

It then builds a small catalogue of what it found, reading the ticker, the date and the number of levels from each file name.

In [3]:
MY_DATA_DIR = None      # e.g. Path("/full/path/to/the/folder/with/the/csv/files") if the search below fails

here = Path.cwd()
candidates = [
    MY_DATA_DIR,
    here / "data",
    here / "LOBdemo25" / "data",
    here / "Microstructure demos" / "LOBdemo25" / "data",
    here.parent / "LOBdemo25" / "data",
    Path.home() / "Desktop" / "COMP0049" / "data",
]
DATA_DIR = next((Path(p) for p in candidates
                 if p is not None and Path(p).is_dir() and any(Path(p).glob("*_message_*.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("No LOBSTER files found. Put the csv files in a folder called 'data' next to this "
                            "notebook, or set MY_DATA_DIR above to the folder that contains them.")

# Catalogue of the LOBSTER files: (ticker, date, levels) -> {'message': path, 'orderbook': path} 

# A regular expression that describes a LOBSTER file name, e.g. "AMZN_2012-06-21_34200000_57600000_message_10.csv".
# Each (?P<name>...) group captures one piece of the name so that we can read it back by name afterwards:
#   ticker  -> letters (and a dot, for tickers like BRK.B)
#   date    -> yyyy-mm-dd
#   \d+_\d+ -> start and end time in milliseconds after midnight (not captured: we don't need them)
#   kind    -> "message" or "orderbook"
#   levels  -> number of price levels in the file (10, 50, ...)
FILE_PATTERN = re.compile(r"^(?P<ticker>[A-Za-z.]+)_(?P<date>\d{4}-\d{2}-\d{2})_\d+_\d+_"
                          r"(?P<kind>message|orderbook)_(?P<levels>\d+)\.csv$")

catalogue = {}
for path in sorted(DATA_DIR.glob("*.csv")):        # every csv file in the data folder
    match = FILE_PATTERN.match(path.name)
    if match:                                       # skip files whose name does not follow the LOBSTER convention
        key = (match["ticker"], match["date"], int(match["levels"]))
        # setdefault: create the entry {} for this (ticker, date, levels) if it does not exist yet, then
        # store the path under "message" or "orderbook" -> the two files of a stock-day end up in the same entry
        catalogue.setdefault(key, {})[match["kind"]] = path

# Print the data folder as a path relative to the notebook if possible (shorter), otherwise the full path
try:
    shown = DATA_DIR.relative_to(here)
except ValueError:                                  # raised when DATA_DIR is not inside the notebook folder
    shown = DATA_DIR
print(f"Data folder: {shown}")

# Summary table: one row per stock day, with the size of the two files in MB (None if a file is missing)
pd.DataFrame([{"ticker": t, "date": d, "levels": n,
               "message file (MB)": round(f["message"].stat().st_size / 1e6, 1) if "message" in f else None,
               "orderbook file (MB)": round(f["orderbook"].stat().st_size / 1e6, 1) if "orderbook" in f else None}
              for (t, d, n), f in catalogue.items()]).sort_values(["date", "ticker"]).reset_index(drop=True)

Data folder: Desktop/COMP0049/data


,ticker,date,levels,message file (MB),orderbook file (MB)
0,AMZN,2012-06-21,10,11.2,63.2
1,MSFT,2012-06-21,10,27.6,170.2
2,BAC,2025-10-06,10,19.9,97.0
3,BKNG,2025-10-06,10,2.4,11.8
4,GOOG,2025-10-06,10,113.3,563.7
5,TSLA,2025-10-06,10,95.2,465.7


In [4]:
def lobster_files(ticker, date=None, levels=None):
    """Return the message and orderbook paths for a ticker (the most recent day if several are available)."""
    # 1. Keep the catalogue entries that match the request. Each key is a tuple (ticker, date, levels),
    #    so k[0] is the ticker, k[1] the date and k[2] the number of levels.
    #    "date is None or ..." means: if no date was asked for, accept any date (same for levels).
    keys = [k for k in catalogue
            if k[0] == ticker and (date is None or k[1] == date) and (levels is None or k[2] == levels)]
    if not keys:                                    # the list is empty: nothing matched
        raise FileNotFoundError(f"No LOBSTER files for {ticker} (date={date}, levels={levels}) in {DATA_DIR}")

    # 2. If several stock days match (e.g. TSLA 2019 and TSLA 2025), take the most recent one:
    #    sort by (date, levels) and keep the last element. Dates as "yyyy-mm-dd" sort correctly as strings.
    key = sorted(keys, key=lambda k: (k[1], k[2]))[-1]

    # 3. Both files must be present. {"message", "orderbook"} <= set(files) reads "is a subset of":
    #    True only if the entry has a path for the message AND for the orderbook file.
    files = catalogue[key]
    if not {"message", "orderbook"} <= set(files):
        raise FileNotFoundError(f"{ticker} {key[1]}: need both the message and the orderbook file")

    # 4. Return everything in one dictionary. **files unpacks {"message": path, "orderbook": path}
    #    into the new dictionary, so the result has the keys ticker, date, levels, message, orderbook.
    return {"ticker": key[0], "date": key[1], "levels": key[2], **files}

---
# Part 1 — The message file

## 1.1 What is in it?

Each row is one **event** that changed the visible book. There are six columns:

| Column | Meaning |
|---|---|
| `Time` | seconds after midnight, with a precision of up to nanoseconds |
| `Type` | type of event (table below) |
| `OrderID` | unique identifier of the limit order involved (no information about the trader) |
| `Size` | number of shares |
| `Price` | dollar price × 10,000 |
| `TradeDirection` | side of the **limit order** involved: `+1` buy (bid side), `-1` sell (ask side) |

| `Type` | Event |
|---|---|
| 1 | submission of a new limit order |
| 2 | cancellation (partial deletion of a limit order) |
| 3 | deletion (total deletion of a limit order) |
| 4 | execution of a **visible** limit order |
| 5 | execution of a **hidden** limit order |
| 6 | cross trade (e.g. the opening auction) |
| 7 | trading halt indicator |

> **Sign convention.** `TradeDirection` is the side of the *resting limit order*. An execution with `TradeDirection = -1` means that a **sell** limit order was hit, i.e. the trade was initiated by a **buyer**. The sign of the market order used in the theory chapters is therefore $\epsilon = -\texttt{TradeDirection}$. 

## 1.2 Loading the data

`load_messages` reads the file, keeps the events between 09:30 and 16:00, and indexes the rows by their timestamp. Files downloaded recently have a seventh, empty column, so we read the first six only. We also keep `row_index`, the position of each row in the raw file: it is what ties the message file to the order book file.

In [7]:
MESSAGE_COLUMNS = ["Time", "Type", "OrderID", "Size", "Price", "TradeDirection"]


def load_messages(ticker, date=None, levels=None, start=START_TRAD, end=END_TRAD):
    """Load a LOBSTER message file, keeping the events between `start` and `end` (seconds after midnight)."""
    files = lobster_files(ticker, date, levels)
    msg = pd.read_csv(files["message"], header=None, names=MESSAGE_COLUMNS, usecols=range(6))

    msg["row_index"] = np.arange(len(msg))            # row number in the raw file -> aligns the order book file
    msg = msg[(msg["Time"] >= start) & (msg["Time"] <= end)].copy()

    # timestamp = trading day + seconds after midnight
    msg.index = pd.Timestamp(files["date"]) + pd.to_timedelta(msg["Time"].to_numpy(), unit="s")
    msg.index.name = "DateTime"
    return msg


STOCKS = ["AMZN", "MSFT"]                              
TICKER_COLORS = dict(zip(STOCKS, STOCK_PALETTE))       # one colour per stock, used in every figure
msg_book = {ticker: load_messages(ticker) for ticker in STOCKS}

> **Question 1.1.** Check that the function has done what we think it should: print the first and the last rows of the two tables.

In [8]:
msg_book["MSFT"].head()

,Time,Type,OrderID,Size,Price,TradeDirection,row_index
DateTime,,,,,,,
2012-06-21 09:30:00.013994120,34200.013994,3,16085616,100,310400,-1,0
2012-06-21 09:30:00.013994120,34200.013994,1,16116348,100,310500,-1,1
2012-06-21 09:30:00.015247805,34200.015248,1,16116658,100,310400,-1,2
2012-06-21 09:30:00.015442111,34200.015442,1,16116704,100,310500,-1,3
2012-06-21 09:30:00.015789148,34200.015789,1,16116752,100,310600,-1,4


In [9]:
msg_book["MSFT"].tail()

,Time,Type,OrderID,Size,Price,TradeDirection,row_index
DateTime,,,,,,,
2012-06-21 15:59:59.861465727,57599.861466,3,287081309,9249,301400,-1,668760
2012-06-21 15:59:59.861566671,57599.861567,3,287117122,783,301400,-1,668761
2012-06-21 15:59:59.873177404,57599.873177,3,287070525,800,301500,-1,668762
2012-06-21 15:59:59.907796528,57599.907797,5,0,100,301350,-1,668763
2012-06-21 15:59:59.907796528,57599.907797,5,0,300,301350,-1,668764


In [10]:
msg_book["AMZN"].head()

,Time,Type,OrderID,Size,Price,TradeDirection,row_index
DateTime,,,,,,,
2012-06-21 09:30:00.017459617,34200.017460,5,0,1,2238200,-1,0
2012-06-21 09:30:00.189607670,34200.189608,1,11885113,21,2238100,1,1
2012-06-21 09:30:00.189607670,34200.189608,1,3911376,20,2239600,-1,2
2012-06-21 09:30:00.189607670,34200.189608,1,11534792,100,2237500,1,3
2012-06-21 09:30:00.189607670,34200.189608,1,1365373,13,2240000,-1,4


In [11]:
msg_book["AMZN"].tail()

,Time,Type,OrderID,Size,Price,TradeDirection,row_index
DateTime,,,,,,,
2012-06-21 15:59:59.872741285,57599.872741,3,286560364,100,2207600,-1,269743
2012-06-21 15:59:59.903989046,57599.903989,3,287142900,100,2206200,-1,269744
2012-06-21 15:59:59.955241980,57599.955242,3,286967592,170,2206900,-1,269745
2012-06-21 15:59:59.958244616,57599.958245,1,287174077,100,2206300,-1,269746
2012-06-21 15:59:59.959359650,57599.959360,3,287174077,100,2206300,-1,269747


> **Question 1.2.** How many events were recorded on this trading day for the two stocks?
>
> **Question 1.3.** Up to which temporal precision are the events recorded (minutes, seconds, …)?

In [14]:
for ticker in STOCKS:
    x = len(msg_book[ticker].index)
    print(f"{ticker}: " f"Events: {x} events | " f"First: {msg_book[ticker].index[0]} | " f"Last: {msg_book[ticker].index[-1]} | "
    f"Events per second: {np.round((x/(END_TRAD - START_TRAD)), 1)}")

AMZN: Events: 269748 events | First: 2012-06-21 09:30:00.017459617 | Last: 2012-06-21 15:59:59.959359650 | Events per second: 11.5
MSFT: Events: 668765 events | First: 2012-06-21 09:30:00.013994120 | Last: 2012-06-21 15:59:59.907796528 | Events per second: 28.6


## 1.3 What happens in a limit order book?

> **Question 1.4.** Write a `summary` function that groups the events by type and, for each type, (1) counts the events and (2) sums the number of shares, both in absolute terms and as a share of the total.
>
> **Question 1.5.** What do you observe about how often the different events occur?

In [45]:
def summary(ticker):
    per_type = msg_book[ticker].groupby("Type")["Size"].agg(["count", "sum"])
    per_type["count (%)"] = np.round(per_type["count"] / sum(per_type["count"])*100, 2)
    per_type["sum (%)"] = np.round(per_type["sum"] / sum(per_type["sum"])*100, 2)
    return per_type 
    
print("AMZN"); display(summary("AMZN"))
print("MSFT"); display(summary("MSFT"))

AMZN


,count,sum,count (%),sum (%)
Type,,,,
1,131954,13027425,48.92,49.86
2,2917,345515,1.08,1.32
3,123458,11946930,45.77,45.72
4,8974,613248,3.33,2.35
5,2445,197507,0.91,0.76


MSFT


,count,sum,count (%),sum (%)
Type,,,,
1,329566,174783168,49.28,50.35
2,5602,2179740,0.84,0.63
3,300183,159320803,44.89,45.90
4,29798,9887669,4.46,2.85
5,3616,936752,0.54,0.27


- **Limit orders dominate.** About half of all events are submissions of new limit orders (49%) and almost as many are cancellations or deletions (46–47%). Only 4–5% of the events are executions.
- **Most limit orders are never executed.** In shares, 94% (AMZN) and 92% (MSFT) of the submitted volume is cancelled or deleted, and there are 10 to 12 new limit orders for every execution.
- **Orders are withdrawn whole.** Total deletions are forty to fifty times more frequent than partial cancellations: an order is removed (and typically re-submitted at another price) rather than reduced.
- **Hidden executions are a minority, but a larger one for AMZN** (0.9% of events against 0.5% for MSFT). 

## 1.4 Trading halts

When trading in a stock is halted, LOBSTER writes a message of type `7`. The state is encoded in the `Price` column, with `TradeDirection = -1` and all other fields set to 0:

| `Price` | Meaning |
|---|---|
| `-1` | trading **halted** |
| `0` | **quoting resumes**, trading still halted |
| `1` | **trading resumes** |

The matching rows of the order book file simply repeat the previous state of the book; the reason for the halt is not recorded. Halts are triggered by news pending, by the Limit Up–Limit Down bands, or by technical problems. NASDAQ publishes the halts and their reason codes at https://www.nasdaqtrader.com/trader.aspx?id=TradeHalts.

> **Question 1.6.** Was trading halted for AMZN or MSFT on this day?

In [49]:
for ticker in STOCKS:
    print(f"Number of trading halts for {ticker}: ", len(msg_book[ticker][msg_book[ticker]["Type"]==7]))

Number of trading halts for AMZN:  0
Number of trading halts for MSFT:  0


---
# Part 2 — The order book file

## 2.1 What is in it?

The order book file has **one row per row of the message file**: the state of the book right *after* the corresponding event. With $N$ levels it has $4N$ columns, ordered as

`Ask Price 1, Ask Size 1, Bid Price 1, Bid Size 1, Ask Price 2, Ask Size 2, Bid Price 2, Bid Size 2, …`

Level 1 is the best quote on each side; prices are again in dollars × 10,000. A level with no orders is filled with the price `9999999999` on the ask side (`-9999999999` on the bid side) and size 0.

`load_orderbook` reads the file, keeps the rows that survived the 09:30–16:00 filter of the message file (this is what `row_index` is for), and returns a dictionary of four aligned tables: ask prices, ask sizes, bid prices and bid sizes. Prices are kept in LOBSTER units; we divide by 10,000 only when we need dollars.

In [23]:
BOOK_TABLES = ["Ask Price", "Ask Size", "Bid Price", "Bid Size"]


def load_orderbook(ticker, msg, date=None, levels=None, chunk_rows=500_000):
    """Load a LOBSTER orderbook file aligned with `msg`. Returns a dict of four DataFrames (one column per level)."""
    files = lobster_files(ticker, date, levels)
    n_levels = files["levels"]
    names = [f"{table} {level}" for level in range(1, n_levels + 1) for table in BOOK_TABLES]
    dtypes = {name: ("int64" if "Price" in name else "int32") for name in names}     # compact types

    # These files are big (up to 2.4 million rows x 40 columns). To keep the memory use low we read them in
    # chunks of rows and split each chunk into the four tables straight away: the columns of a table are
    # every 4th column of the file, starting at column 0, 1, 2 or 3.
    pieces = {table: [] for table in BOOK_TABLES}
    for chunk in pd.read_csv(files["orderbook"], header=None, names=names, dtype=dtypes, chunksize=chunk_rows):
        for i, table in enumerate(BOOK_TABLES):
            pieces[table].append(chunk.iloc[:, i::4].copy())
    book = {table: pd.concat(pieces.pop(table)) for table in BOOK_TABLES}

    # keep exactly the rows kept in the message file, and give them the same time index
    n_rows = len(book["Ask Price"])
    assert n_rows > msg["row_index"].max(), "message and orderbook files do not have the same number of rows"
    for table in BOOK_TABLES:
        if n_rows != len(msg):
            book[table] = book[table].iloc[msg["row_index"].to_numpy()]
        book[table].index = msg.index
    return book


OB = {ticker: load_orderbook(ticker, msg_book[ticker]) for ticker in STOCKS}

OB
├── "AMZN"
│   ├── "Ask Price"  → DataFrame, 269,748 rows × 10 columns: Ask Price 1 … Ask Price 10
│   ├── "Ask Size"   → DataFrame, 269,748 rows × 10 columns: Ask Size 1 … Ask Size 10
│   ├── "Bid Price"  → DataFrame, 269,748 rows × 10 columns: Bid Price 1 … Bid Price 10
│   └── "Bid Size"   → DataFrame, 269,748 rows × 10 columns: Bid Size 1 … Bid Size 10
└── "MSFT"
    ├── "Ask Price"  → 668,765 rows × 10
    ├── "Ask Size"
    ├── "Bid Price"
    └── "Bid Size"

The output is a dictionary of tables. Each row is the book after one event, so every table must have exactly as many rows as the message table. Let us check.

In [24]:
def data_checker(book, msg, ticker=""):
    for table, values in book.items():
        assert values.shape[0] == msg.shape[0], f"{ticker} {table}: {values.shape[0]} rows, expected {msg.shape[0]}"
        print(f"ok!  {ticker:5s} {table:10s} {values.shape}")


for ticker in STOCKS:
    data_checker(OB[ticker], msg_book[ticker], ticker)

ok!  AMZN  Ask Price  (269748, 10)
ok!  AMZN  Ask Size   (269748, 10)
ok!  AMZN  Bid Price  (269748, 10)
ok!  AMZN  Bid Size   (269748, 10)
ok!  MSFT  Ask Price  (668765, 10)
ok!  MSFT  Ask Size   (668765, 10)
ok!  MSFT  Bid Price  (668765, 10)
ok!  MSFT  Bid Size   (668765, 10)


In [25]:
# So the raw 40 column orderbook file (Ask Price 1, Ask Size 1, Bid Price 1, Bid Size 1, Ask Price 2, …) has simply been cut into four 10 column tables: 

OB["AMZN"]["Ask Price"]                 # all ask prices, all events
OB["AMZN"]["Ask Price"]["Ask Price 1"]  # the best ask through the day (a Series)
OB["AMZN"]["Bid Size"].iloc[1000]       # the ten bid volumes right after event number 1000
OB["AMZN"]["Bid Size"].loc["2012-06-21 11:00"]  # same, for the events in that minute

,Bid Size 1,Bid Size 2,Bid Size 3,Bid Size 4,Bid Size 5,Bid Size 6,Bid Size 7,Bid Size 8,Bid Size 9,Bid Size 10
DateTime,,,,,,,,,,
2012-06-21 11:00:00.133492026,100,300,20,20,26,126,400,201,200,214
2012-06-21 11:00:00.560059285,100,300,20,20,26,126,400,201,100,200
2012-06-21 11:00:00.565628076,100,300,20,20,26,126,400,201,200,200
2012-06-21 11:00:00.679919711,100,300,20,20,26,126,400,201,200,200
2012-06-21 11:00:00.680113853,100,300,20,20,26,126,400,201,200,200
...,...,...,...,...,...,...,...,...,...,...
2012-06-21 11:00:58.458086853,144,18,99,300,100,114,100,300,200,200
2012-06-21 11:00:59.269221949,144,18,99,300,100,114,100,300,200,200
2012-06-21 11:00:59.269226192,144,18,99,300,100,114,100,300,200,200


> **Question 2.1.** Look at the structure of the data: what do the rows and the columns of each table represent?

In [50]:
OB["AMZN"]["Bid Price"].head()

,Bid Price 1,Bid Price 2,Bid Price 3,Bid Price 4,Bid Price 5,Bid Price 6,Bid Price 7,Bid Price 8,Bid Price 9,Bid Price 10
DateTime,,,,,,,,,,
2012-06-21 09:30:00.017459617,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000,2189700
2012-06-21 09:30:00.189607670,2238100,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000
2012-06-21 09:30:00.189607670,2238100,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000
2012-06-21 09:30:00.189607670,2238100,2237500,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500
2012-06-21 09:30:00.189607670,2238100,2237500,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500


In [26]:
OB["AMZN"]["Bid Price"].head()

,Bid Price 1,Bid Price 2,Bid Price 3,Bid Price 4,Bid Price 5,Bid Price 6,Bid Price 7,Bid Price 8,Bid Price 9,Bid Price 10
DateTime,,,,,,,,,,
2012-06-21 09:30:00.017459617,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000,2189700
2012-06-21 09:30:00.189607670,2238100,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000
2012-06-21 09:30:00.189607670,2238100,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500,2202000
2012-06-21 09:30:00.189607670,2238100,2237500,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500
2012-06-21 09:30:00.189607670,2238100,2237500,2231800,2230700,2230400,2230000,2226200,2213000,2204000,2202500


## 2.2 Best quotes, spread, mid-price and gaps

From the first two levels we can compute, for every event, the quantities defined in the lecture notes (Definition 2.5 and §2.5):

- best bid $b(t)$ and best ask $a(t)$, and the volumes available there (the **depth at the best**);
- the **bid–ask spread** $s(t) = a(t) - b(t)$;
- the **mid-price** $x(t) = \tfrac12 [a(t) + b(t)]$;
- the **first gap** $d_1$ on each side: the distance between the best and the second-best price.

> **Question 2.2.** Compute the bid, ask, mid-price, spread and first gaps associated with each event.

In [53]:
def summary(ticker):
    df = pd.DataFrame({
    "Best Bid": OB[ticker]["Bid Price"]["Bid Price 1"],
    "Best Ask": OB[ticker]["Ask Price"]["Ask Price 1"],
    "Bid-Ask Spread": OB[ticker]["Ask Price"]["Ask Price 1"] - OB[ticker]["Bid Price"]["Bid Price 1"],
    "Mid-Price": (OB[ticker]["Bid Price"]["Bid Price 1"] + OB[ticker]["Ask Price"]["Ask Price 1"]) / 2,
    "First-Gap (Bid Side)": OB[ticker]["Bid Price"]["Bid Price 1"] - OB[ticker]["Bid Price"]["Bid Price 2"],
    "First-Gap (Ask Side)": OB[ticker]["Ask Price"]["Ask Price 2"] - OB[ticker]["Ask Price"]["Ask Price 1"]
    })
    return df

print("AMZN"); display(summary("AMZN").head())
print("MSFT"); display(summary("MSFT").head())

AMZN


,Best Bid,Best Ask,Bid-Ask Spread,Mid-Price,First-Gap (Bid Side),First-Gap (Ask Side)
DateTime,,,,,,
2012-06-21 09:30:00.017459617,2231800,2239500,7700,2235650.0,1100,400
2012-06-21 09:30:00.189607670,2238100,2239500,1400,2238800.0,6300,400
2012-06-21 09:30:00.189607670,2238100,2239500,1400,2238800.0,6300,100
2012-06-21 09:30:00.189607670,2238100,2239500,1400,2238800.0,600,100
2012-06-21 09:30:00.189607670,2238100,2239500,1400,2238800.0,600,100


MSFT


,Best Bid,Best Ask,Bid-Ask Spread,Mid-Price,First-Gap (Bid Side),First-Gap (Ask Side)
DateTime,,,,,,
2012-06-21 09:30:00.013994120,309500,309900,400,309700.0,200,600
2012-06-21 09:30:00.013994120,309500,309900,400,309700.0,200,600
2012-06-21 09:30:00.015247805,309500,309900,400,309700.0,200,500
2012-06-21 09:30:00.015442111,309500,309900,400,309700.0,200,500
2012-06-21 09:30:00.015789148,309500,309900,400,309700.0,200,500
